# 02. Target Definition and potential leakage treatment

**Objective**

This notebook formalizes the modelling target, reviews whether the available outcome can be used as a bad flag, investigates vintage / maturity bias, defines the modelling population and removes post-origination leakage variables.

The dataset is treated as an application credit risk dataset with a historical final loan outcome target:

- **Bad / default proxy:** `Charged Off`
- **Good:** `Fully Paid`

Important limitation: the dataset does not provide a formally defined observation window and performance window. Therefore, the target is interpreted as a **historical lifetime outcome**, not as a regulatory 12-month IRB/IFRS 9 PD target.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Project paths and load raw data

The notebook is designed to run from the `notebooks/` folder. It will also work if executed from the project root.

In [ ]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "raw" / "loan_processed_data.csv").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data" / "raw" / "loan_processed_data.csv").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "loan_processed_data.csv"
if not RAW_PATH.exists():
    alt_path = Path("/mnt/data/loan_processed_data.csv")
    if alt_path.exists():
        RAW_PATH = alt_path
    else:
        raise FileNotFoundError("Could not find loan_processed_data.csv. Expected it in data/raw/.")

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "02.target_definition_and_leakage"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data path: {RAW_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print(df_raw.shape)
df_raw.head()

## 2. Target variable inspection

The target is derived from `loan_status`. Since only two final statuses are present, the variable can be used to construct a binary bad flag.

In [ ]:
TARGET_COL = "loan_status"
BAD_STATUS = "Charged Off"
GOOD_STATUS = "Fully Paid"

if TARGET_COL not in df_raw.columns:
    raise KeyError(f"{TARGET_COL} was not found in the dataset.")

status_distribution = (
    df_raw[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .reset_index(name="count")
)
status_distribution["share"] = status_distribution["count"] / len(df_raw)
status_distribution

In [ ]:
unexpected_statuses = sorted(set(df_raw[TARGET_COL].dropna().unique()) - {BAD_STATUS, GOOD_STATUS})
missing_target_count = df_raw[TARGET_COL].isna().sum()

print(f"Unexpected statuses: {unexpected_statuses}")
print(f"Missing target values: {missing_target_count:,}")

In [ ]:
df = df_raw.copy()
df["target_bad"] = np.where(df[TARGET_COL].eq(BAD_STATUS), 1,
                            np.where(df[TARGET_COL].eq(GOOD_STATUS), 0, np.nan))

target_summary = pd.DataFrame({
    "metric": ["total_records", "good_count", "bad_count", "observed_bad_rate", "missing_target_count", "unexpected_status_count"],
    "value": [
        len(df),
        int((df["target_bad"] == 0).sum()),
        int((df["target_bad"] == 1).sum()),
        float(df["target_bad"].mean()),
        int(df["target_bad"].isna().sum()),
        len(unexpected_statuses)
    ]
})
target_summary

### Target interpretation

`target_bad` is a binary historical loan outcome flag:

- `1` = Charged Off
- `0` = Fully Paid

This is suitable for a credit default classification project. However, it should not be presented as a formal regulatory 12-month PD target because no explicit observation and performance windows are available in the dataset.

## 3. Date parsing and lifecycle duration checks

We use `issue_d` as the loan origination / disbursement date proxy. `last_pymnt_d` is post-origination information and is used here only for target/population diagnostics, not as a model predictor.

In [ ]:
DATE_COLS = ["issue_d", "last_pymnt_d", "last_credit_pull_d", "earliest_cr_line"]

for col in DATE_COLS:
    if col in df.columns:
        df[col + "_parsed"] = pd.to_datetime(df[col], errors="coerce", format="mixed")

if "issue_d_parsed" not in df.columns:
    raise KeyError("issue_d is required for vintage analysis and was not found.")

date_summary_rows = []
for col in DATE_COLS:
    parsed_col = col + "_parsed"
    if parsed_col in df.columns:
        date_summary_rows.append({
            "date_variable": col,
            "missing_count": int(df[parsed_col].isna().sum()),
            "min_date": df[parsed_col].min(),
            "max_date": df[parsed_col].max(),
            "unique_months": int(df[parsed_col].dt.to_period("M").nunique())
        })

date_summary = pd.DataFrame(date_summary_rows)
date_summary

In [ ]:
if {"issue_d_parsed", "last_pymnt_d_parsed"}.issubset(df.columns):
    df["months_on_book_proxy"] = ((df["last_pymnt_d_parsed"] - df["issue_d_parsed"]).dt.days / 30.4375)
    df["duration_days_proxy"] = (df["last_pymnt_d_parsed"] - df["issue_d_parsed"]).dt.days
else:
    df["months_on_book_proxy"] = np.nan
    df["duration_days_proxy"] = np.nan

duration_by_status = (
    df.groupby(TARGET_COL)["duration_days_proxy"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .reset_index()
)
duration_by_status

In [ ]:
negative_duration_records = df[df["duration_days_proxy"] < 0].copy()
negative_duration_summary = pd.DataFrame({
    "metric": ["negative_duration_count", "negative_duration_pct"],
    "value": [len(negative_duration_records), len(negative_duration_records) / len(df)]
})
negative_duration_summary

### Duration interpretation

The lifecycle duration proxy is used only as a diagnostic. It confirms that `loan_status` behaves like a final loan outcome: charged-off loans generally have shorter observed duration than fully paid loans.

Negative durations are marked as data quality issues. They are not used as model inputs.

## 4. Vintage / maturity bias analysis

We inspect observed bad rate by issue quarter. Recent vintages with too little time to reach charge-off can show artificially low bad rates.

In [ ]:
df["issue_month"] = df["issue_d_parsed"].dt.to_period("M").astype(str)
df["issue_quarter"] = df["issue_d_parsed"].dt.to_period("Q").astype(str)
df["issue_year"] = df["issue_d_parsed"].dt.year

vintage_quarter_summary = (
    df.groupby("issue_quarter", dropna=False)
    .agg(
        total_accounts=("target_bad", "size"),
        good_count=("target_bad", lambda x: int((x == 0).sum())),
        bad_count=("target_bad", lambda x: int((x == 1).sum())),
        observed_bad_rate=("target_bad", "mean"),
        median_duration_days=("duration_days_proxy", "median"),
        mean_duration_days=("duration_days_proxy", "mean")
    )
    .reset_index()
)

vintage_quarter_summary

In [ ]:
vintage_year_summary = (
    df.groupby("issue_year", dropna=False)
    .agg(
        total_accounts=("target_bad", "size"),
        good_count=("target_bad", lambda x: int((x == 0).sum())),
        bad_count=("target_bad", lambda x: int((x == 1).sum())),
        observed_bad_rate=("target_bad", "mean"),
        median_duration_days=("duration_days_proxy", "median"),
        mean_duration_days=("duration_days_proxy", "mean")
    )
    .reset_index()
)

vintage_year_summary

### Maturity bias observation

The latest vintages, especially `2018Q3` and `2018Q4`, show abnormally low observed bad rates relative to previous vintages. This is consistent with insufficient seasoning time between loan origination and the data extraction/snapshot date.

For model development, these latest vintages are excluded from the modelling population.

In [ ]:
EXCLUDED_MATURITY_BIAS_QUARTERS = ["2018Q3", "2018Q4"]

maturity_bias_exclusion = vintage_quarter_summary[
    vintage_quarter_summary["issue_quarter"].isin(EXCLUDED_MATURITY_BIAS_QUARTERS)
].copy()

maturity_bias_exclusion["exclusion_reason"] = (
    "Excluded due to likely maturity bias / insufficient performance window seasoning"
)

maturity_bias_exclusion

In [ ]:
df["population_exclusion_reason"] = np.nan

df.loc[df["issue_quarter"].isin(EXCLUDED_MATURITY_BIAS_QUARTERS), "population_exclusion_reason"] = (
    "Maturity bias: 2018Q3-Q4 excluded due to abnormally low observed bad rate and insufficient seasoning"
)

df.loc[df["target_bad"].isna(), "population_exclusion_reason"] = "Missing or unexpected target status"

population_definition = (
    df["population_exclusion_reason"]
    .fillna("Included in modelling population")
    .value_counts()
    .rename_axis("population_status")
    .reset_index(name="record_count")
)
population_definition["share"] = population_definition["record_count"] / len(df)
population_definition

In [ ]:
df_model_population = df[df["population_exclusion_reason"].isna()].copy()

population_comparison = pd.DataFrame({
    "population": ["raw_dataset", "model_population_after_maturity_filter"],
    "records": [len(df), len(df_model_population)],
    "good_count": [int((df["target_bad"] == 0).sum()), int((df_model_population["target_bad"] == 0).sum())],
    "bad_count": [int((df["target_bad"] == 1).sum()), int((df_model_population["target_bad"] == 1).sum())],
    "observed_bad_rate": [float(df["target_bad"].mean()), float(df_model_population["target_bad"].mean())],
    "min_issue_quarter": [df["issue_quarter"].min(), df_model_population["issue_quarter"].min()],
    "max_issue_quarter": [df["issue_quarter"].max(), df_model_population["issue_quarter"].max()]
})

population_comparison

## 5. Leakage review

The following variables are reviewed for leakage risk. Variables related to repayments, recoveries, final settlement, or post-origination bureau pulls are excluded from model training.

In [ ]:
leakage_candidates = {
    "loan_status": ("target", "Outcome variable used to construct target_bad"),
    "total_pymnt": ("drop_leakage", "Total payments received after origination; post-outcome information"),
    "total_rec_int": ("drop_leakage", "Interest received after origination; post-origination performance information"),
    "total_rec_late_fee": ("drop_leakage", "Late fees received after origination; direct repayment behavior"),
    "recoveries": ("drop_leakage", "Recovery amount is only known after default/collections"),
    "last_pymnt_d": ("drop_leakage", "Last payment date is post-origination performance information"),
    "last_pymnt_amnt": ("drop_leakage", "Last payment amount is post-origination performance information"),
    "last_credit_pull_d": ("drop_leakage", "Post-origination credit pull date"),
    "last_fico_range_high": ("drop_leakage", "Post-origination/latest FICO information"),
    "debt_settlement_flag": ("drop_leakage", "Debt settlement is known after deterioration/default"),
    "issue_d": ("keep_for_split_not_predictor", "Origination/disbursement date proxy; keep for vintage split, not as model predictor"),
    "earliest_cr_line": ("keep_for_feature_engineering", "Can be used to engineer credit history length relative to issue date"),
    "emp_title": ("candidate_exclude_or_encode_later", "High-cardinality text field; not leakage but requires careful treatment"),
    "title": ("candidate_exclude_or_encode_later", "High-cardinality text/application title field; requires careful treatment"),
    "zip_code": ("candidate_exclude_or_encode_later", "Geographic field; may raise stability/fairness concerns and requires careful treatment")
}

leakage_review = []
for col, (role, reason) in leakage_candidates.items():
    leakage_review.append({
        "variable": col,
        "present_in_dataset": col in df_model_population.columns,
        "recommended_role": role,
        "reason": reason
    })

leakage_review = pd.DataFrame(leakage_review)
leakage_review

In [ ]:
LEAKAGE_DROP_COLS = [
    col for col, (role, reason) in leakage_candidates.items()
    if role in ["target", "drop_leakage"] and col in df_model_population.columns
]

TECHNICAL_HELPER_COLS = [c for c in df_model_population.columns if c.endswith("_parsed")]
TECHNICAL_HELPER_COLS += ["months_on_book_proxy", "duration_days_proxy", "population_exclusion_reason"]
TECHNICAL_HELPER_COLS = [c for c in TECHNICAL_HELPER_COLS if c in df_model_population.columns]

print("Leakage/target columns to drop from modelling predictors:")
print(LEAKAGE_DROP_COLS)
print("Technical helper columns to remove from modelling predictors:")
print(TECHNICAL_HELPER_COLS)

## 6. Create clean initial modelling dataset

The modelling dataset keeps:

- `target_bad`
- application / origination variables
- `issue_d`, `issue_month`, `issue_quarter`, `issue_year` for time-based splitting and diagnostics

It removes post-origination leakage variables and technical helper columns.

In [ ]:
df_model = df_model_population.copy()

drop_from_model_dataset = sorted(set(LEAKAGE_DROP_COLS + TECHNICAL_HELPER_COLS))
df_model = df_model.drop(columns=drop_from_model_dataset, errors="ignore")

model_dataset_summary = pd.DataFrame({
    "metric": [
        "model_dataset_records",
        "model_dataset_columns",
        "target_bad_count",
        "target_good_count",
        "observed_bad_rate",
        "min_issue_quarter",
        "max_issue_quarter"
    ],
    "value": [
        len(df_model),
        df_model.shape[1],
        int((df_model["target_bad"] == 1).sum()),
        int((df_model["target_bad"] == 0).sum()),
        float(df_model["target_bad"].mean()),
        df_model["issue_quarter"].min(),
        df_model["issue_quarter"].max()
    ]
})

model_dataset_summary

In [ ]:
model_columns = pd.DataFrame({
    "variable": df_model.columns,
    "dtype": [str(df_model[c].dtype) for c in df_model.columns],
    "missing_count": [int(df_model[c].isna().sum()) for c in df_model.columns],
    "missing_pct": [float(df_model[c].isna().mean()) for c in df_model.columns],
    "unique_count": [int(df_model[c].nunique(dropna=False)) for c in df_model.columns]
})
model_columns

## 7. Save notebook artifacts

The notebook saves:

- Excel report with target, vintage, population and leakage tables
- Pickle artifact dictionary
- Initial cleaned modelling dataset as `.pkl`

In [ ]:
artifacts = {
    "target_summary": target_summary,
    "status_distribution": status_distribution,
    "date_summary": date_summary,
    "duration_by_status": duration_by_status,
    "negative_duration_summary": negative_duration_summary,
    "vintage_quarter_summary": vintage_quarter_summary,
    "vintage_year_summary": vintage_year_summary,
    "maturity_bias_exclusion": maturity_bias_exclusion,
    "population_definition": population_definition,
    "population_comparison": population_comparison,
    "leakage_review": leakage_review,
    "model_dataset_summary": model_dataset_summary,
    "model_columns": model_columns,
    "excluded_maturity_bias_quarters": EXCLUDED_MATURITY_BIAS_QUARTERS,
    "leakage_drop_cols": LEAKAGE_DROP_COLS,
    "technical_helper_cols": TECHNICAL_HELPER_COLS,
    "drop_from_model_dataset": drop_from_model_dataset,
    "notes": {
        "target_definition": "target_bad = 1 if loan_status == Charged Off, else 0 if Fully Paid",
        "target_limitation": "Historical lifetime outcome, not a formal regulatory 12-month PD target",
        "population_filter": "2018Q3 and 2018Q4 excluded due to likely maturity bias / insufficient seasoning",
        "issue_d_interpretation": "issue_d used as origination/disbursement date proxy"
    }
}

pkl_artifact_path = OUTPUT_DIR / "target_definition_and_leakage_artifacts.pkl"
model_pkl_path = OUTPUT_DIR / "initial_modeling_dataset.pkl"
excel_path = OUTPUT_DIR / "target_definition_and_leakage_report.xlsx"

with open(pkl_artifact_path, "wb") as f:
    pickle.dump(artifacts, f)

df_model.to_pickle(model_pkl_path)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    target_summary.to_excel(writer, sheet_name="target_summary", index=False)
    status_distribution.to_excel(writer, sheet_name="status_distribution", index=False)
    date_summary.to_excel(writer, sheet_name="date_summary", index=False)
    duration_by_status.to_excel(writer, sheet_name="duration_by_status", index=False)
    negative_duration_summary.to_excel(writer, sheet_name="negative_duration", index=False)
    vintage_quarter_summary.to_excel(writer, sheet_name="vintage_quarter_raw", index=False)
    vintage_year_summary.to_excel(writer, sheet_name="vintage_year_raw", index=False)
    maturity_bias_exclusion.to_excel(writer, sheet_name="maturity_exclusion", index=False)
    population_definition.to_excel(writer, sheet_name="population_definition", index=False)
    population_comparison.to_excel(writer, sheet_name="population_comparison", index=False)
    leakage_review.to_excel(writer, sheet_name="leakage_review", index=False)
    model_dataset_summary.to_excel(writer, sheet_name="model_dataset_summary", index=False)
    model_columns.to_excel(writer, sheet_name="model_columns", index=False)

print("Saved files:")
print(f"- {pkl_artifact_path}")
print(f"- {model_pkl_path}")
print(f"- {excel_path}")

## 8. Final conclusion

The modelling target is accepted as a binary historical outcome flag:

- Bad = `Charged Off`
- Good = `Fully Paid`

The latest issue vintages `2018Q3` and `2018Q4` are excluded due to likely maturity bias. The resulting modelling population is more appropriate for downstream credit risk modelling, WOE/IV analysis, logistic regression scorecard development and OOT validation.